In [1]:
import pandas as pd
import numpy as np


In [ ]:
pd_validacion = pd.read_csv("validacion.csv")
y_val = np.array(pd_validacion["klass"])


In [ ]:
pd_Gemma = pd.read_csv("LLM_probs_umbral_0.89_app.csv")
probs_Gemma = np.array(pd_Gemma['klass'])
pd_Gemma_1 = pd.read_csv("LLM_8791_validacion.csv")
probs_Gemma_1 = np.array(pd_Gemma_1['klass'])
pd_betito = pd.read_csv("robertuito_probs_3.1e-05_565_158_0.001278_32_0.153572_0.136504_val.csv")
probs_betito = np.array(pd_betito['klass'])
pd_Osth = pd.read_csv("Qwen_OSTH_finetuned_scores_val.csv")
probs_Osth = np.array(pd_Osth["score_qwen"])
pd_beto = pd.read_csv("bert_scores_val.csv")
probs_beto = np.array(pd_beto["score_bert"])


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# p_roberta = probabilidades de clase 1 que guardaste de RoBERTuito (array de floats 0-1)
# test_scores = los scores que ya tienes de Gemma V2

p_gemma_1   = probs_Gemma_1
p_roberta = probs_betito

# Búsqueda de pesos óptimos en validación
best_w, best_f1 = 0.5, 0.0
for w in np.arange(0.1, 1.0, 0.05):
    p_ens = w * p_gemma_1 + (1 - w) * p_roberta
    # también busca umbral óptimo para cada peso
    for thresh in np.arange(0.3, 0.7, 0.02):
        preds = (p_ens >= thresh).astype(int)
        f1 = f1_score(y_val, preds, average='macro')
        if f1 > best_f1:
            best_f1, best_w, best_thresh = f1, w, thresh

print(f"Mejor peso Gemma: {best_w:.2f}  Umbral: {best_thresh:.2f}  F1 val: {best_f1:.4f}")

Mejor peso Gemma: 0.70  Umbral: 0.68  F1 val: 0.8804


In [40]:
import numpy as np
from sklearn.metrics import f1_score

# 1. Tus arrays de probabilidades de la Clase 1 (asegúrate de que todos tengan la misma longitud)
p_gemma_old = probs_Osth      # Tu modelo Gemma anterior
p_gemma_new = probs_Gemma_1    # Tu nuevo Gemma 3 optimizado
p_roberta   = probs_betito   # RoBERTuito

# Variables para guardar la mejor combinación encontrada en validación
best_weights = (0.33, 0.33, 0.34)
best_thresh = 0.5
best_f1 = 0.0

print("Buscando la combinación óptima de pesos y umbral...")

# 2. Grid Search de Pesos (Paso de 0.05)
for w_gemma_old in np.arange(0.0, 1.01, 0.05):
    for w_gemma_new in np.arange(0.0, 1.01 - w_gemma_old, 0.05):

        # El tercer peso se calcula automáticamente para que sumen 1.0
        w_roberta = 1.0 - w_gemma_old - w_gemma_new

        # Pequeño control por flotantes imprecisos de Python
        if w_roberta < 0.0:
            continue

        # 3. Calculamos la probabilidad combinada del ensamble para este intento
        p_ens = (w_gemma_old * p_gemma_old) + (w_gemma_new * p_gemma_new) + (w_roberta * p_roberta)

        # 4. Búsqueda del umbral óptimo de clasificación para esta combinación de pesos
        for thresh in np.arange(0.3, 0.71, 0.02):
            preds = (p_ens >= thresh).astype(int)

            # NOTA: Asegúrate de que 'y_val' corresponda exactamente al mismo orden de tus arrays de probs
            f1 = f1_score(y_val, preds, average='macro')

            # Si encontramos un F1-Score más alto, guardamos la configuración
            if f1 > best_f1:
                best_f1 = f1
                best_weights = (w_gemma_old, w_gemma_new, w_roberta)
                best_thresh = thresh

# ── RESULTADOS FINALES ───────────────────────────────────────────────────────
w_old, w_new, w_rob = best_weights
print("\n¡Búsqueda completada con éxito! 🚀")
print("─" * 50)
print(f"Pesos óptimos encontrados:")
print(f"  • Gemma Anterior (p_gemma):   {w_old:.2f}")
print(f"  • Gemma Nuevo (p_gemma_1):   {w_new:.2f}")
print(f"  • RoBERTuito (p_roberta):     {w_rob:.2f}")
print(f"  (Suma total de pesos: {w_old + w_new + w_rob:.2f})")
print("─" * 50)
print(f"Umbral de decisión (Threshold): {best_thresh:.2f}")
print(f"Mejor Macro F1-Score en Val:    {best_f1:.4f}")
print("─" * 50)

Buscando la combinación óptima de pesos y umbral...

¡Búsqueda completada con éxito! 🚀
──────────────────────────────────────────────────
Pesos óptimos encontrados:
  • Gemma Anterior (p_gemma):   0.35
  • Gemma Nuevo (p_gemma_1):   0.50
  • RoBERTuito (p_roberta):     0.15
  (Suma total de pesos: 1.00)
──────────────────────────────────────────────────
Umbral de decisión (Threshold): 0.64
Mejor Macro F1-Score en Val:    0.8899
──────────────────────────────────────────────────


In [60]:
import numpy as np
from sklearn.metrics import f1_score

# =============================================================================
# 1. CARGA DE PROBABILIDADES (Tus 4 modelos)
# =============================================================================
p_gemma_old = probs_beto       # Gemma anterior
p_gemma_new = probs_Gemma_1     # Tu nuevo Gemma 3 optimizado
p_roberta   = probs_betito      # RoBERTuito
p_qwen_osth = probs_Osth        # ¡El nuevo Qwen amaestrado con teoría OSTH!

# Variables para guardar la mejor configuración histórica
best_weights = (0.25, 0.25, 0.25, 0.25)
best_thresh = 0.5
best_f1 = 0.0

print("Buscando la combinación óptima de pesos (4 modelos) y umbral...")

# =============================================================================
# 2. GRID SEARCH EN 4D (Paso de 0.05 para velocidad y precisión)
# =============================================================================
for w_gemma_old in np.arange(0.0, 1.01, 0.05):
    for w_gemma_new in np.arange(0.0, 1.01 - w_gemma_old, 0.05):
        
        # Guardamos un acumulado parcial; si ya llegamos a 1.0, los bucles internos no tienen sentido
        suma_parcial_1 = w_gemma_old + w_gemma_new
        
        for w_roberta in np.arange(0.0, 1.01 - suma_parcial_1, 0.05):
            
            # El cuarto peso (Qwen OSTH) se calcula automáticamente por diferencia
            w_qwen_osth = 1.0 - w_gemma_old - w_gemma_new - w_roberta
            
            # Control estricto para evitar flotantes negativos debido a la precisión de Python
            if w_qwen_osth < -1e-9:
                continue
            if w_qwen_osth < 0.0:
                w_qwen_osth = 0.0  # Corrección de redondeo menor

            # 3. Calculamos la probabilidad combinada del ensamble de 4 vías
            p_ens = (
                (w_gemma_old * p_gemma_old) + 
                (w_gemma_new * p_gemma_new) + 
                (w_roberta * p_roberta) + 
                (w_qwen_osth * p_qwen_osth)
            )

            # 4. Búsqueda del umbral óptimo de decisión
            for thresh in np.arange(0.3, 0.71, 0.02):
                preds = (p_ens >= thresh).astype(int)

                # RECUERDA: 'y_val' debe estar en el mismo orden que las probabilidades
                f1 = f1_score(y_val, preds, average='macro')

                # Si encontramos un nuevo récord de F1, guardamos la alineación
                if f1 > best_f1:
                    best_f1 = f1
                    best_weights = (w_gemma_old, w_gemma_new, w_roberta, w_qwen_osth)
                    best_thresh = thresh

# =============================================================================
# ── RESULTADOS FINALES ───────────────────────────────────────────────────────
# =============================================================================
w_old, w_new, w_rob, w_qwen = best_weights
print("\n¡Búsqueda de 4 vías completada con éxito! 🏆")
print("─" * 55)
print(f"Pesos óptimos encontrados:")
print(f"  • Gemma Anterior (p_gemma):    {w_old:.2f}")
print(f"  • Gemma Nuevo (p_gemma_1):    {w_new:.2f}")
print(f"  • RoBERTuito (p_roberta):      {w_rob:.2f}")
print(f"  • Qwen OSTH (p_qwen_osth):     {w_qwen:.2f}")
print(f"  (Suma total de pesos: {w_old + w_new + w_rob + w_qwen:.2f})")
print("─" * 55)
print(f"Umbral de decisión (Threshold):  {best_thresh:.2f}")
print(f"Mejor Macro F1-Score en Val:     {best_f1:.4f}")
print("─" * 55)

Buscando la combinación óptima de pesos (4 modelos) y umbral...

¡Búsqueda de 4 vías completada con éxito! 🏆
───────────────────────────────────────────────────────
Pesos óptimos encontrados:
  • Gemma Anterior (p_gemma):    0.25
  • Gemma Nuevo (p_gemma_1):    0.30
  • RoBERTuito (p_roberta):      0.10
  • Qwen OSTH (p_qwen_osth):     0.35
  (Suma total de pesos: 1.00)
───────────────────────────────────────────────────────
Umbral de decisión (Threshold):  0.46
Mejor Macro F1-Score en Val:     0.8927
───────────────────────────────────────────────────────


In [65]:
pd_Gemma_test = pd.read_csv("LLM_probs_0.54_app_test.csv")
test_Gemma = np.array(pd_Gemma_test['klass'])
pd_Gemma_1_test = pd.read_csv("LLM_8791_test.csv")
test_Gemma_1 = np.array(pd_Gemma_1_test['klass'])
pd_betito_test = pd.read_csv("robertuito_probs_3.1e-05_565_158_0.001278_32_0.153572_0.136504_test.csv")
test_betito = np.array(pd_betito_test['klass'])
pd_Osth_test = pd.read_csv("Qwen_OSTH_finetuned_scores_test_V2.csv")
test_Osth = np.array(pd_Osth_test["score_qwen"])
pd_beto_test = pd.read_csv("bert_scores_test.csv")
beto_test = np.array(pd_beto_test["score_bert"])

In [ ]:
import numpy as np
import pandas as pd

# Tus parámetros óptimos encontrados en val
best_w      = 0.70
best_thresh = 0.68

p_gemma_test = test_Gemma
p_gemma_test_1 = test_Gemma_1
p_roberta_test = test_betito

# Ensemble sobre test
p_ensemble_test = best_w * p_gemma_test_1 + (1 - best_w) * p_roberta_test

# Predicciones finales
test_preds_ensemble = (p_ensemble_test >= best_thresh).astype(int)

# Distribución (verifica que no esté colapsado)
unique, counts = np.unique(test_preds_ensemble, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

# Guardar CSV

pd.DataFrame({
    'id'    : range(1, len(test_preds_ensemble) + 1),
    'klass' : test_preds_ensemble,
    #'score' : p_ensemble_test,
}).to_csv('GG_Robertuito_V5.csv', index=False)

print('✅ GG_Robertuito_V5.csv listo')

In [67]:
p_gemma_test = beto_test
p_gemma_test_1 = test_Gemma_1
p_roberta_test = test_betito
p_qwen_osth = test_Osth


p_ens_test = (best_weights[0] * p_gemma_test) + (best_weights[1] * p_gemma_test_1) + (best_weights[2] * p_roberta_test) + (best_weights[3] * p_qwen_osth)
preds_finales_test = (p_ens_test >= best_thresh).astype(int)

unique, counts = np.unique(preds_finales_test, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

# Guardar CSV

NAME = 'BETO_GG_Robertuito_WW_V1.csv'

pd.DataFrame({
    'id'    : range(1, len(preds_finales_test) + 1),
    'klass' : preds_finales_test,
    #'score' : p_ensemble_test,
}).to_csv(NAME, index=False)

print(f'✅ {NAME} listo')

{0: 3402, 1: 2198}
✅ BETO_GG_Robertuito_WW_V1.csv listo


In [53]:
# Ejecuta esto para entender qué tan lejos estás del techo
print(f"Pesos óptimos: {best_weights}")
print(f"Umbral: {best_thresh:.3f}")

# Correlación entre modelos (diversidad del ensamble)
from scipy.stats import pearsonr
modelos = {
    'Gemma':      p_gemma_test,
    'Gemma_1':    p_gemma_test_1,
    'RoBERTuito': p_roberta_test,
    'Qwen_OSTH':  p_qwen_osth,
}
import pandas as pd
scores_df = pd.DataFrame(modelos)
print("\nCorrelación entre modelos (menor = más diversidad = mejor ensamble):")
print(scores_df.corr().round(3))

Pesos óptimos: (np.float64(0.05), np.float64(0.45), np.float64(0.15000000000000002), np.float64(0.3499999999999999))
Umbral: 0.640

Correlación entre modelos (menor = más diversidad = mejor ensamble):
            Gemma  Gemma_1  RoBERTuito  Qwen_OSTH
Gemma       1.000    0.933       0.844      0.924
Gemma_1     0.933    1.000       0.821      0.896
RoBERTuito  0.844    0.821       1.000      0.841
Qwen_OSTH   0.924    0.896       0.841      1.000


In [56]:
import cma

p_gemma_val = probs_Gemma
p_gemma_val_1 = probs_Gemma_1
p_roberta_val = probs_betito
p_qwen_val = probs_Osth
val_labels = y_val
def objective(params):
    """CMA-ES minimiza → devolvemos 1 - F1_macro"""
    w  = np.array(params[:4])
    w  = np.abs(w) / np.abs(w).sum()   # normalizar a simplex
    th = float(np.clip(params[4], 0.3, 0.8))
    
    p_ens = (w[0]*p_gemma_val + w[1]*p_gemma_val_1 +
             w[2]*p_roberta_val + w[3]*p_qwen_val)
    preds = (p_ens >= th).astype(int)
    return 1.0 - f1_score(val_labels, preds, average='macro')

# Punto de inicio: tus mejores pesos actuales
x0     = list(best_weights) + [best_thresh]
sigma0 = 0.15   # exploración inicial

es = cma.CMAEvolutionStrategy(x0, sigma0, {
    'maxiter'  : 500,
    'tolx'     : 1e-6,
    'tolfun'   : 1e-6,
    'seed'     : 42,
    'verbose'  : 1,
})
es.optimize(objective)

w_opt = np.abs(es.result.xbest[:4])
w_opt = w_opt / w_opt.sum()
th_opt = float(np.clip(es.result.xbest[4], 0.3, 0.8))
print(f"Pesos CMA-ES: {w_opt}")
print(f"Umbral CMA-ES: {th_opt:.4f}")

(4_w,8)-aCMA-ES (mu_w=2.6,w_1=52%) in dimension 5 (seed=42, Thu Jun  4 16:29:16 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      8 1.150923958534544e-01 1.0e+00 1.23e-01  1e-01  1e-01 0:00.0
    2     16 1.142921082498122e-01 1.3e+00 1.05e-01  9e-02  1e-01 0:00.0
    3     24 1.150923958534544e-01 1.4e+00 9.09e-02  7e-02  9e-02 0:00.0
   29    232 1.094101769499208e-01 4.5e+00 9.27e-03  3e-03  7e-03 0:00.3
termination on {'tolflatfitness': 1}
final/bestever f-value = 1.094102e-01 1.094102e-01 after 232/102 evaluations
incumbent solution: [0.03873719, 0.41808499, 0.08245487, 0.28183148, 0.63396399]
std deviations: [0.00727181, 0.00685328, 0.00305348, 0.00726742, 0.00255651]
Pesos CMA-ES: [0.0365175  0.51905866 0.09246359 0.35196026]
Umbral CMA-ES: 0.6295


In [41]:

p_gemma_test = test_Gemma
p_gemma_test_1 = test_Gemma_1
p_roberta_test = test_betito
p_qwen_osth = test_Osth


p_ens_test = (best_weights[0] * p_gemma_test) + (best_weights[1] * p_gemma_test_1) + (best_weights[2] * p_roberta_test)
preds_finales_test = (p_ens_test >= best_thresh).astype(int)

unique, counts = np.unique(preds_finales_test, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

# Guardar CSV

NAME = 'WW_GG_Robertuito_V1.csv'

pd.DataFrame({
    'id'    : range(1, len(preds_finales_test) + 1),
    'klass' : preds_finales_test,
    #'score' : p_ensemble_test,
}).to_csv(NAME, index=False)

print(f'✅ {NAME} listo')

{0: 3484, 1: 2116}
✅ WW_GG_Robertuito_V1.csv listo
